# Insurance Premium Prediction

**Tabular Regression Project** — Predict insurance premium costs.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Medical Cost Personal (Kaggle, 1,338 rows)
Target: `charges`

## 2 · Project Overview

We predict **insurance charges** from demographic and health features. This clean, well-understood dataset (1,338 rows × 7 columns) is a classic regression benchmark where one feature — **smoking status** — dominates the target.

## 3 · Learning Objectives

1. Work with a clean, small regression dataset.
2. Discover feature dominance (smoker status).
3. Create interaction features.
4. Handle skewed targets.
5. Compare models on a dataset with clear non-linear patterns.

## 4 · Problem Statement

Predict **annual insurance charges** from age, sex, BMI, number of children, smoking status, and region.

## 5 · Why This Project Matters

- Insurance pricing is a core actuarial ML application.
- Teaches how a single dominant feature (smoker) can drive predictions.
- Demonstrates the value of interaction features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: mirichoi0218/insurance |
| **Rows** | 1,338 |
| **Features** | age, sex, bmi, children, smoker, region |
| **Target** | charges (USD) |

## 7 · Dataset Source and License Notes

- **Source**: Medical Cost Personal Datasets (Kaggle).
- **License**: Public domain.
- **Note**: From a statistics textbook — clean and well-understood.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'charges'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [4]:
import kagglehub, glob
try:
    path = kagglehub.dataset_download('mirichoi0218/insurance')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR
csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f'Loaded: {df.shape}')
df.head()

Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\mirichoi0218\insurance\versions\1
Loaded: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 12 · Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Target mean: ${df[TARGET].mean():,.0f}')

Missing: 0
Duplicates: 1
Target mean: $13,270


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df[TARGET].hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Charges Distribution')
axes[0,1].scatter(df['age'], df[TARGET], c=df['smoker'].map({'yes':1,'no':0}), alpha=0.4, cmap='coolwarm')
axes[0,1].set_xlabel('Age'); axes[0,1].set_ylabel('Charges')
axes[0,1].set_title('Age vs Charges (red=smoker)')
df.boxplot(column=TARGET, by='smoker', ax=axes[1,0])
axes[1,0].set_title('Charges by Smoker'); plt.suptitle('')
axes[1,1].scatter(df['bmi'], df[TARGET], alpha=0.3)
axes[1,1].set_xlabel('BMI'); axes[1,1].set_ylabel('Charges')
axes[1,1].set_title('BMI vs Charges')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

count     1338.000000
mean     13270.422265
std      12110.011237
min       1121.873900
25%       4740.287150
50%       9382.033000
75%      16639.912515
max      63770.428010
Name: charges, dtype: float64

Skewness: 1.52


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [8]:
from sklearn.preprocessing import LabelEncoder
for col in ['sex', 'smoker', 'region']:
    df[col] = LabelEncoder().fit_transform(df[col])
print(f'Preprocessed: {df.shape}')

Preprocessed: (1338, 7)


## 17 · Feature Engineering

In [9]:
df['smoker_bmi'] = df['smoker'] * df['bmi']
df['smoker_age'] = df['smoker'] * df['age']
df['age_bmi'] = df['age'] * df['bmi']

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (1070, 9), Test: (268, 9)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:,.0f}, MAE={baseline_mae:,.0f}, R²={baseline_r2:.4f}')

Baseline LR: RMSE=4,571, MAE=2,773, R²=0.8654


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                            Adjusted R-Squared  R-Squared         RMSE  \
Model                                                                    
CatBoostRegressor                     0.875178   0.879386  4327.259899   
GradientBoostingRegressor             0.873128   0.877405  4362.652916   
RandomForestRegressor                 0.865769   0.870293  4487.403279   
LGBMRegressor                         0.861783   0.866442  4553.541776   
SGDRegressor                          0.861711   0.866373  4554.721754   
Ridge                                 0.861254   0.865931  4562.247957   
LarsCV                                0.861174   0.865853  4563.563821   
BayesianRidge                         0.861046   0.865730  4565.658395   
LassoLarsIC                           0.861003   0.865689  4566.363252   
LassoCV                               0.860981   0.865667  4566.728303   
LassoLarsCV                           0.860955   0.865642  4567.157581   
RidgeCV                               

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

CatBoost: RMSE=4361.44, MAE=2411.65, R²=0.8775


LightGBM: RMSE=4613.39, MAE=2667.42, R²=0.8629


XGBoost: RMSE=4610.56, MAE=2494.70, R²=0.8631


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  CatBoost              RMSE=4361.44  MAE=2411.65  R²=0.8775
  Baseline_LR           RMSE=4570.53  MAE=2772.71  R²=0.8654
  XGBoost               RMSE=4610.56  MAE=2494.70  R²=0.8631
  LightGBM              RMSE=4613.39  MAE=2667.42  R²=0.8629

Top 3: ['CatBoost', 'Baseline_LR', 'XGBoost']


## 23 · Final Eval of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

Top 3 Final Results:
CatBoost: RMSE=4361.44, MAE=2411.65, R²=0.8775
Baseline_LR: RMSE=4570.53, MAE=2772.71, R²=0.8654
XGBoost: RMSE=4610.56, MAE=2494.70, R²=0.8631


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Smoker status** is overwhelmingly the strongest predictor (3-4× higher charges).
- **smoker_bmi** interaction captures the amplified risk of smoking + high BMI.
- **Age** is the second-strongest driver.
- Insurance companies use these patterns for risk-based pricing.

## 26 · Limitations

- Only 1,338 rows.
- Binary smoker status oversimplifies (pack-years matter).
- No pre-existing conditions or medical history.
- Synthetic/textbook data.

## 27 · How to Improve

1. Log-transform charges.
2. Add health history features.
3. Granular smoking data.
4. Separate models for smokers and non-smokers.

## 28 · Production Considerations

- Regulatory requirements for insurance pricing.
- Anti-discrimination laws.
- Model transparency requirements.
- Actuarial validation needed.

## 29 · Common Mistakes

1. Not creating smoker × BMI interaction.
2. Ignoring the bimodal distribution.
3. Using linear models without interactions.
4. Not recognizing smoker dominance.

## 30 · Mini Challenge

1. Build separate models for smokers vs non-smokers.
2. Try log(charges) as target.
3. Add BMI categories.
4. Use SHAP to explain predictions.

## 31 · Final Summary

- Smoker status dominates insurance cost prediction.
- Interaction features are critical for linear models.
- Boosting models learn interactions automatically.
- Clean dataset ideal for regression fundamentals.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
